<a href="https://colab.research.google.com/github/Joyanne05/vres-sentiment-analysis/blob/main/sentiment_analysis_sbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
df = pd.read_csv("course_data_clean.csv")
df = df[["reviews", "course_rating_int"]]
df.head()

,reviews,course_rating_int
0,go to office hours and practice,1.0
1,One of my least favourite courses. Although th...,0.0
2,It starts with a very low pace but after midte...,0.0
3,Took this in 2018 with no programming experien...,1.0
4,I loved everything about cs 115. Great instruc...,1.0


In [12]:
print(df["course_rating_int"].value_counts(dropna=False))

course_rating_int
1.0    9786
0.0    4830
NaN     222
Name: count, dtype: int64


In [13]:
# Remove rows missing a review or rating
df = df.dropna(subset=["reviews", "course_rating_int"]).copy()

# Remove empty or whitespace-only reviews
df = df[df["reviews"].str.strip().ne("")].copy()

# Convert labels to integer
df["course_rating_int"] = df["course_rating_int"].astype(int)

# Check the result
print(df["course_rating_int"].value_counts(dropna=False))
print("Remaining rows:", len(df))

course_rating_int
1    9784
0    4824
Name: count, dtype: int64
Remaining rows: 14608


In [14]:
# Split dataset (80/20)
X = df["reviews"]
y = df["course_rating_int"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
from sentence_transformers import SentenceTransformer

# Vectorisation using SBERT
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding training reviews...")
X_train_sbert = sbert_model.encode(X_train.tolist(), show_progress_bar=True)

print("Encoding test reviews...")
X_test_sbert = sbert_model.encode(X_test.tolist(), show_progress_bar=True)

print("Training matrix (SBERT):", X_train_sbert.shape)
print("Test matrix (SBERT):", X_test_sbert.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding training reviews...


Batches:   0%|          | 0/366 [00:00<?, ?it/s]

Encoding test reviews...


Batches:   0%|          | 0/92 [00:00<?, ?it/s]

Training matrix (SBERT): (11686, 384)
Test matrix (SBERT): (2922, 384)


In [16]:
# Logistic Regression on SBERT embeddings
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_sbert, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [17]:
y_pred = lr_model.predict(X_test_sbert)

In [18]:
from sklearn.metrics import accuracy_score, classification_report

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

Accuracy: 0.770
              precision    recall  f1-score   support

    Negative       0.63      0.74      0.68       965
    Positive       0.86      0.78      0.82      1957

    accuracy                           0.77      2922
   macro avg       0.74      0.76      0.75      2922
weighted avg       0.78      0.77      0.77      2922



In [19]:
# Gaussian Naive Bayes is used here because MultinomialNB requires non-negative feature values,
# whereas SBERT embeddings contain both positive and negative continuous dimensions.
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_sbert, y_train)

nb_pred = nb_model.predict(X_test_sbert)

In [20]:
from sklearn.metrics import accuracy_score, classification_report

print(f"Accuracy: {accuracy_score(y_test, nb_pred):.3f}")

print(classification_report(
    y_test,
    nb_pred,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

Accuracy: 0.736
              precision    recall  f1-score   support

    Negative       0.59      0.65      0.62       965
    Positive       0.82      0.78      0.80      1957

    accuracy                           0.74      2922
   macro avg       0.70      0.71      0.71      2922
weighted avg       0.74      0.74      0.74      2922



In [21]:
# Support Vector Machine on SBERT embeddings
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

svm_model = LinearSVC(class_weight='balanced', random_state=42)
svm_model.fit(X_train_sbert, y_train)

svm_pred = svm_model.predict(X_test_sbert)

print(f"Accuracy: {accuracy_score(y_test, svm_pred):.3f}")

print(classification_report(
    y_test,
    svm_pred,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

Accuracy: 0.767
              precision    recall  f1-score   support

    Negative       0.62      0.75      0.68       965
    Positive       0.86      0.78      0.82      1957

    accuracy                           0.77      2922
   macro avg       0.74      0.76      0.75      2922
weighted avg       0.78      0.77      0.77      2922



In [22]:
import pandas as pd

baseline_results = pd.DataFrame({
    "model": ["Logistic Regression", "Gaussian Naive Bayes", "Linear SVM"],
    "accuracy": [0.770, 0.736, 0.767],
    "macro_f1": [0.75, 0.71, 0.75],
    "negative_recall": [0.74, 0.65, 0.75]
})

baseline_results.to_csv("baseline_results.csv", index=False)
baseline_results

,model,accuracy,macro_f1,negative_recall
0,Logistic Regression,0.770,0.75,0.74
1,Gaussian Naive Bayes,0.736,0.71,0.65
2,Linear SVM,0.767,0.75,0.75


## Cross validation

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.metrics import make_scorer, recall_score
from sklearn.base import BaseEstimator, TransformerMixin
from sentence_transformers import SentenceTransformer

# Custom transformer to wrap SBERT into scikit-learn pipelines
class SBERTVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, 'tolist'):
            X = X.tolist()
        return self.model.encode(X, show_progress_bar=False)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [24]:
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": GaussianNB(),
    "Linear SVM": LinearSVC(random_state=42)
}

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "negative_recall": make_scorer(
        recall_score,
        pos_label=0,
        zero_division=0
    )
}

In [25]:
cv_results = {}
summary = []

for name, model in models.items():
    pipeline = Pipeline([
        ("sbert", SBERTVectorizer(model_name='all-MiniLM-L6-v2')),
        ("clf", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,  # Keep sequential to avoid thread contention during PyTorch encoding
        error_score="raise"
    )

    cv_results[name] = scores

    summary.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "macro_f1_mean": scores["test_macro_f1"].mean(),
        "macro_f1_std": scores["test_macro_f1"].std(),
        "negative_recall_mean": scores["test_negative_recall"].mean()
    })

cv_summary = pd.DataFrame(summary)
print("5-fold cross-validation with SBERT fixed model settings")
display(cv_summary.round(3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

... (10 outputs left out)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

5-fold cross-validation with SBERT fixed model settings


,model,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,negative_recall_mean
0,Logistic Regression,0.781,0.004,0.733,0.006,0.540
1,Naive Bayes,0.719,0.006,0.691,0.007,0.637
2,Linear SVM,0.781,0.007,0.736,0.009,0.557


In [26]:
from sklearn.model_selection import GridSearchCV

# Param grids adjusted for classifiers on SBERT embeddings (removing TF-IDF parameters)
param_grids = {
    "Logistic Regression": {
        "clf__C": [0.1, 1, 10],
        "clf__class_weight": [None, "balanced"]
    },

    "Naive Bayes": {
        "clf__var_smoothing": [1e-9, 1e-8, 1e-7]
    },

    "Linear SVM": {
        "clf__C": [0.1, 1, 10],
        "clf__class_weight": [None, "balanced"]
    }
}

In [27]:
searches = {}
tuning_summary = []

for name, model in models.items():
    pipeline = Pipeline([
        ("sbert", SBERTVectorizer(model_name='all-MiniLM-L6-v2')),
        ("clf", model)
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring=scoring,
        refit="macro_f1",
        cv=cv,
        n_jobs=1,  # Sequential to avoid parallel PyTorch conflicts
        verbose=1,
        error_score="raise"
    )

    search.fit(X_train, y_train)
    searches[name] = search

    best_index = search.best_index_
    results = search.cv_results_

    tuning_summary.append({
        "model": name,
        "cv_accuracy": results["mean_test_accuracy"][best_index],
        "cv_macro_f1": results["mean_test_macro_f1"][best_index],
        "cv_macro_f1_std": results["std_test_macro_f1"][best_index],
        "cv_negative_recall": results["mean_test_negative_recall"][best_index],
        "best_params": search.best_params_
    })

    print(f"\n{name}")
    print("Best settings:", search.best_params_)
    print(f"Best CV macro-F1: {search.best_score_:.3f}")

tuned_results = pd.DataFrame(tuning_summary)
display(tuned_results.drop(columns="best_params").round(3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fitting 5 folds for each of 6 candidates, totalling 30 fits


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

... (81 outputs left out)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Linear SVM
Best settings: {'clf__C': 10, 'clf__class_weight': None}
Best CV macro-F1: 0.737


,model,cv_accuracy,cv_macro_f1,cv_macro_f1_std,cv_negative_recall
0,Logistic Regression,0.755,0.736,0.004,0.733
1,Naive Bayes,0.719,0.691,0.007,0.637
2,Linear SVM,0.780,0.737,0.009,0.567


In [28]:
from sklearn.model_selection import cross_val_predict

best_lr = searches["Logistic Regression"].best_estimator_

oof_pred = cross_val_predict(
    best_lr,
    X_train,       # Raw review text
    y_train,
    cv=cv,         # Your existing five-fold splitter
    method="predict",
    n_jobs=-1
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
error_analysis = pd.DataFrame({
    "review": X_train.to_numpy(),
    "actual": y_train.to_numpy(),
    "predicted": oof_pred
})

mistakes = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
].copy()

print("Number of misclassified reviews:", len(mistakes))

Number of misclassified reviews: 2864


In [30]:
sample = mistakes.sample(
    n=min(10, len(mistakes)),
    random_state=42
)

for index, row in sample.iterrows():
    print("Row:", index)
    print("Review:", row["review"])
    print("Actual label:", row["actual"])
    print("Model prediction:", row["predicted"])
    print()

Row: 6603
Review: ewwwww
Actual label: 0
Model prediction: 1

Row: 7296
Review: A challenging course that will probably be useful in future terms.
Actual label: 0
Model prediction: 1

Row: 3302
Review: When they warn you that you can get lost easily in this course, they REALLY aren't kidding. Without practice and a full understanding, tensors can quickly turn into a wailing and shrieking Lovecraftian horror. Review of NE 112 is practically needed whether or not one remembers NE 112 content.
Actual label: 1
Model prediction: 0

Row: 7223
Review: I took this class under the impression that it would be a fairly qualitative course. This was not the case. There was a ton of stats and other complex math that I had struggled with throughout the term. I'm surprised there are no math pre-requisites.
Actual label: 1
Model prediction: 0

Row: 1721
Review: This course had some interesting assignments, such as exploiting a buffer overflow, a format string vulnerability, and finding holes in a poorl

In [31]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report
)

test_results = []
test_predictions = {}

for name, search in searches.items():
    best_model = search.best_estimator_

    # Pass raw review text, not X_test_tfidf
    predictions = best_model.predict(X_test)
    test_predictions[name] = predictions

    test_results.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, predictions),
        "test_macro_f1": f1_score(
            y_test, predictions, average="macro"
        ),
        "test_negative_recall": recall_score(
            y_test, predictions, pos_label=0, zero_division=0
        )
    })

    print(f"\n{name}")
    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Negative", "Positive"],
        zero_division=0
    ))

test_summary = pd.DataFrame(test_results)
display(test_summary.round(3))


Logistic Regression
              precision    recall  f1-score   support

    Negative       0.62      0.75      0.68       965
    Positive       0.86      0.78      0.82      1957

    accuracy                           0.77      2922
   macro avg       0.74      0.76      0.75      2922
weighted avg       0.78      0.77      0.77      2922


Naive Bayes
              precision    recall  f1-score   support

    Negative       0.59      0.65      0.62       965
    Positive       0.82      0.78      0.80      1957

    accuracy                           0.74      2922
   macro avg       0.70      0.71      0.71      2922
weighted avg       0.74      0.74      0.74      2922


Linear SVM
              precision    recall  f1-score   support

    Negative       0.74      0.56      0.64       965
    Positive       0.81      0.90      0.85      1957

    accuracy                           0.79      2922
   macro avg       0.77      0.73      0.74      2922
weighted avg       0.78     

,model,test_accuracy,test_macro_f1,test_negative_recall
0,Logistic Regression,0.766,0.747,0.745
1,Naive Bayes,0.736,0.708,0.647
2,Linear SVM,0.789,0.745,0.564
